In [42]:
import pandas as pd
import numpy as np

train_df = pd.read_csv("train_data.csv")
test_df = pd.read_csv("test_data.csv")

In [43]:
X_train = train_df.drop(columns=["actual_productivity", "MeetsTarget"])
y_reg_train = train_df["actual_productivity"].to_numpy()
y_clf_train = train_df["MeetsTarget"].to_numpy()

X_test = test_df.drop(columns=["actual_productivity", "MeetsTarget"])
y_reg_test = test_df["actual_productivity"].to_numpy()
y_clf_test = test_df["MeetsTarget"].to_numpy()

In [44]:
numerical_features = [
    "targeted_productivity",
    "smv",
    "wip",
    "overtime_per_worker",
    "incentive",
    "idle_time",
    "idle_men",
    "no_of_style_change",
    "no_of_workers"
]

categorical_features = [
    "quarter",
    "department",
    "day",
    "team"
]

In [45]:
wip_median = X_train["wip"].median()

X_train["wip"] = X_train["wip"].fillna(wip_median)
X_test["wip"] = X_test["wip"].fillna(wip_median)

In [46]:
X_train["team"] = X_train["team"].astype(str)
X_test["team"] = X_test["team"].astype(str)

In [47]:
X_train_cat = pd.get_dummies(
    X_train[categorical_features],
    dtype=float
)

X_test_cat = pd.get_dummies(
    X_test[categorical_features],
    dtype=float
)

X_test_cat = X_test_cat.reindex(
    columns=X_train_cat.columns,
    fill_value=0
)

In [48]:
print("Train categorical shape:", X_train_cat.shape)
print("Test categorical shape :", X_test_cat.shape)

Train categorical shape: (957, 25)
Test categorical shape : (240, 25)


In [49]:
print(X_train[categorical_features].dtypes)

quarter       object
department    object
day           object
team          object
dtype: object


In [50]:
X_train_num = X_train[numerical_features].copy()
X_test_num = X_test[numerical_features].copy()

In [51]:
train_mean = X_train_num.mean()
train_std = X_train_num.std(ddof=0)

In [52]:
X_train_num = (X_train_num - train_mean) / train_std
X_test_num = (X_test_num - train_mean) / train_std

In [53]:
X_train_final = pd.concat(
    [X_train_num, X_train_cat],
    axis=1
)

X_test_final = pd.concat(
    [X_test_num, X_test_cat],
    axis=1
)

In [54]:
print("Train shape:", X_train_final.shape)
print("Test shape :", X_test_final.shape)

Train shape: (957, 34)
Test shape : (240, 34)


In [55]:
X_train_final = X_train_final.to_numpy()
X_test_final = X_test_final.to_numpy()


## linear regression

In [56]:
X_train_reg = np.c_[
    np.ones(X_train_final.shape[0]),
    X_train_final
]

X_test_reg = np.c_[
    np.ones(X_test_final.shape[0]),
    X_test_final
]

print(X_train_reg.shape)
print(X_test_reg.shape)

(957, 35)
(240, 35)


In [57]:
import time
y_train = y_reg_train



start_time = time.perf_counter()

w = np.linalg.solve(
    X_train_reg.T @ X_train_reg,
    X_train_reg.T @ y_train
)

manual_reg_train_time = time.perf_counter() - start_time


start_time = time.perf_counter()

y_reg_pred_manual = X_test_reg @ w

manual_reg_prediction_time = time.perf_counter() - start_time


print("Training time   :", manual_reg_train_time, "seconds")
print("Prediction time :", manual_reg_prediction_time, "seconds")

Training time   : 0.008882701000402449 seconds
Prediction time : 5.1021997933276e-05 seconds


In [58]:
errors = y_reg_test - y_reg_pred_manual

mae_manual = np.mean(np.abs(errors))

mse_manual = np.mean(errors ** 2)
rmse_manual = np.sqrt(mse_manual)

ss_res = np.sum(errors ** 2)
ss_tot = np.sum((y_reg_test - np.mean(y_reg_test)) ** 2)

r2_manual = 1 - (ss_res / ss_tot)

print("MAE :", mae_manual)
print("RMSE:", rmse_manual)
print("R²  :", r2_manual)

MAE : 0.1107871608687142
RMSE: 0.14989951191433906
R²  : 0.15375606811636622


## logistic regression

In [59]:
X_train_clf = np.c_[
    np.ones(X_train_final.shape[0]),
    X_train_final
]

X_test_clf = np.c_[
    np.ones(X_test_final.shape[0]),
    X_test_final
]

y_train_clf = y_clf_train
y_test_clf = y_clf_test

print(X_train_clf.shape)
print(X_test_clf.shape)

(957, 35)
(240, 35)


In [60]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

In [61]:
learning_rate = 0.01
iterations = 10000

# Training time
start_time = time.perf_counter()

w_clf = np.zeros(X_train_clf.shape[1])

for i in range(iterations):
    z = X_train_clf @ w_clf
    probabilities = sigmoid(z)

    gradient = (
        X_train_clf.T @ (probabilities - y_train_clf)
    ) / len(y_train_clf)

    w_clf -= learning_rate * gradient

manual_clf_train_time = time.perf_counter() - start_time


# Prediction time
start_time = time.perf_counter()

y_clf_prob_manual = sigmoid(X_test_clf @ w_clf)
y_clf_pred_manual = (y_clf_prob_manual >= 0.5).astype(int)

manual_clf_prediction_time = time.perf_counter() - start_time


print("Training time   :", manual_clf_train_time, "seconds")
print("Prediction time :", manual_clf_prediction_time, "seconds")

Training time   : 0.20310647299993434 seconds
Prediction time : 8.36870021885261e-05 seconds


In [62]:
y_clf_prob_manual = sigmoid(X_test_clf @ w_clf)

print(y_clf_prob_manual[:5])

[0.87686769 0.83205498 0.59215819 0.6777295  0.46705176]


In [63]:
y_clf_pred_manual = (y_clf_prob_manual >= 0.5).astype(int)

print(y_clf_pred_manual[:20])
print("Number of predictions:", len(y_clf_pred_manual))

[1 1 1 1 0 1 0 1 1 1 1 1 1 1 1 1 1 1 1 1]
Number of predictions: 240


In [64]:
tp = np.sum((y_test_clf == 1) & (y_clf_pred_manual == 1))
tn = np.sum((y_test_clf == 0) & (y_clf_pred_manual == 0))
fp = np.sum((y_test_clf == 0) & (y_clf_pred_manual == 1))
fn = np.sum((y_test_clf == 1) & (y_clf_pred_manual == 0))

accuracy = (tp + tn) / len(y_test_clf)

precision = tp / (tp + fp)

recall = tp / (tp + fn)

f1 = 2 * precision * recall / (precision + recall)

print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1-score :", f1)

Accuracy : 0.775
Precision: 0.7887323943661971
Recall   : 0.9491525423728814
F1-score : 0.8615384615384616
